In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline
from scipy.optimize import minimize, brentq
import calendar
from sklearn.linear_model import LinearRegression

In [2]:
data = pd.read_excel("A4Data.xlsx", sheet_name = None)
type(data)

dict

In [3]:
df_yc    = data["YieldCurve"].copy()
df_bonds = data["BondsToPrice"].copy()
df_hist  = data["Historical"].copy()

In [4]:
## datetime conversion
for col in ["Date", "Maturity"]:
    df_yc[col] = pd.to_datetime(df_yc[col])

for col in ["Date", "Maturity", "First Call Date", "Second Call Date", "Third Call Date"]:
    if col in df_bonds.columns:
        df_bonds[col] = pd.to_datetime(df_bonds[col])

df_hist["Date"]     = pd.to_datetime(df_hist["Date"])
df_hist["Maturity"] = pd.to_datetime(df_hist["Maturity"])

In [5]:
def tenor(start, end):
    """30/360 tenor in years."""
    start = pd.Timestamp(start)
    end   = pd.Timestamp(end)
    num = (360 * (end.year - start.year)
           + 30 * (end.month - start.month)
           + (end.day - start.day))
    return num / 360.0

In [6]:
df_yc["Tenor"] = df_yc.apply(lambda r: tenor(r["Date"], r["Maturity"]), axis=1)

## PART 1

In [8]:
## NSS SPot curve
def nss_spot(t, beta0, beta1, beta2, beta3, tau1, tau2):
    t = np.asarray(t, dtype=float)
    def f(x, tau):
        x = np.asarray(x, dtype=float)
        z = x / tau
        num = 1 - np.exp(-z)
        den = np.where(z == 0, 1e-8, z)
        return num / den
    f1 = f(t, tau1)
    f2 = f(t, tau2)
    r = (beta0
         + beta1 * f1
         + beta2 * (f1 - np.exp(-t / tau1))
         + beta3 * (f2 - np.exp(-t / tau2)))
    return r

In [9]:
## obejctive function for scipy optimize
def nss_objective(params, tenor_vec, yield_vec):
    beta0, beta1, beta2, beta3, tau1, tau2 = params
    model = nss_spot(tenor_vec, beta0, beta1, beta2, beta3, tau1, tau2)
    resid = model - yield_vec
    return np.sum(resid**2)

In [10]:
x0 = np.array([5.0, 0.5, 25.0, -25.0, 1.0, 1.0])

In [11]:
res = minimize(nss_objective, x0, args=(df_yc["Tenor"].values, df_yc["Yield"].values), method="BFGS")

In [12]:
beta0, beta1, beta2, beta3, tau1, tau2 = res.x

In [13]:
print("\n=== NSS parameters ===")
print("beta0, beta1, beta2, beta3, tau1, tau2 =", beta0, beta1, beta2, beta3, tau1, tau2)


=== NSS parameters ===
beta0, beta1, beta2, beta3, tau1, tau2 = 2.288800828630904 -2.258942612932206 33.40235919029585 -36.73198764892895 1.1302325301490928 1.16980629092807


In [14]:
## coupon dates
def generate_coupon_dates(start, maturity):
    start    = pd.Timestamp(start)
    maturity = pd.Timestamp(maturity)
    dates = []
    target_day = min(30, maturity.day)
    y, m = start.year, start.month
    while True:
        last_day = calendar.monthrange(y, m)[1]
        d = pd.Timestamp(year=y, month=m, day=min(target_day, last_day))
        if d >= start:
            dates.append(d)
        if d >= maturity:
            break
        if m == 12:
            y, m = y + 1, 1
        else:
            m += 1
    return pd.DatetimeIndex(dates)

In [15]:
## vanila bonds pricing
def price_bond_noncall(row, z_bps):
    face   = 100.0
    start  = pd.Timestamp(row["Date"])
    mat    = pd.Timestamp(row["Maturity"])
    c_rate = row["Coupon"] / 100.0
    z      = z_bps / 10000.0    # spread in decimal
    pay_dates = generate_coupon_dates(start, mat)
    pv   = 0.0
    prev = start
    for d in pay_dates:
        t    = tenor(start, d)
        dtau = tenor(prev,  d)
        spot_pct = nss_spot(t, beta0, beta1, beta2, beta3, tau1, tau2)
        spot     = spot_pct / 100.0  # converting "percent" to decimal
        r  = spot + z
        df = 1.0 / ((1.0 + r)**t)
        cf = face * c_rate * dtau
        if d == pay_dates[-1]:
            cf += face
        pv   += cf * df
        prev  = d
    return pv

In [16]:
## Solving for z spreads
def solve_z_spread(row):
    target = row["Price"]
    def f(z_bps):
        return price_bond_noncall(row, z_bps) - target
    ## Root-finding using Brent's method which is much robust and accurate than newton raphson and bisection
    lo, hi = -1000.0, 2000.0
    f_lo, f_hi = f(lo), f(hi)
    if f_lo * f_hi > 0:
        res = minimize_scalar(lambda z: f(z)**2, bounds=(lo, hi), method="bounded")
        return res.x
    else:
        return brentq(f, lo, hi)

In [17]:
## Z spreads
df_bonds["Z_Spread_bps"] = df_bonds.apply(solve_z_spread, axis=1)
df_bonds[["Ticker", "Z_Spread_bps"]]

,Ticker,Z_Spread_bps
0,AMCX,301.390501
1,ANF,525.442661
2,CRL,266.831906
3,HWM,197.360040
4,JWN,230.406951
5,MTCHII,305.082114
6,TFX,259.155731
7,URI,345.516241


In [18]:
## calculating forward rates
def build_forward_rates(t_vec, spot_vec):
    t_vec   = np.asarray(t_vec, float)
    spot    = np.asarray(spot_vec, float)
    fwd     = np.zeros_like(spot)
    if len(t_vec) == 0:
        return fwd
    fwd[0] = spot[0]  # first period ≈ first spot
    for i in range(1, len(t_vec)):
        t1, t2 = t_vec[i - 1], t_vec[i]
        r1, r2 = spot[i - 1], spot[i]
        dt     = t2 - t1
        fwd[i] = r2 if dt <= 0 else (r2 * t2 - r1 * t1) / dt
    return fwd

In [19]:
## Pricing callable bonds
def price_callable_forward(row, spread_bps):
    face   = 100.0
    start  = pd.Timestamp(row["Date"])
    mat    = pd.Timestamp(row["Maturity"])
    c_rate = row["Coupon"] / 100.0
    spread = spread_bps / 10000.0   # decimal
    pay_dates = generate_coupon_dates(start, mat)
    t_vec     = np.array([tenor(start, d) for d in pay_dates], float)

    dt        = np.empty_like(t_vec)
    dt[0]     = t_vec[0]
    dt[1:]    = t_vec[1:] - t_vec[:-1]

    spot_pct  = nss_spot(t_vec, beta0, beta1, beta2, beta3, tau1, tau2)
    spot      = spot_pct / 100.0          # decimal
    fwd_rf    = build_forward_rates(t_vec, spot)
    fwd_total = fwd_rf + spread

    # coupon cash-flows
    cf = face * c_rate * dt
    cf[-1] += face

    # call schedule
    call_dates  = []
    call_prices = []
    for label in ["First", "Second", "Third"]:
        dcol = f"{label} Call Date"
        pcol = f"{label} Call Price"
        if dcol in row and pd.notna(row[dcol]):
            call_dates.append(pd.Timestamp(row[dcol]).normalize())
            call_prices.append(float(row[pcol]))

    call_dict = dict(zip(call_dates, call_prices))

    # backward induction
    V_next = 0.0
    for i in range(len(pay_dates) - 1, -1, -1):
        V_curr = (V_next + cf[i]) / (1.0 + fwd_total[i] * dt[i])
        d_norm = pay_dates[i].normalize()

        # issuer chooses the cheaper option at call dates
        if d_norm in call_dict:
            V_curr = min(V_curr, call_dict[d_norm])

        V_next = V_curr

    return V_next

In [20]:
## solving for OAS using same brentq as used in Zspread
def solve_oas(row):
    target = row["Price"]
    def f(oas_bps):
        return price_callable_forward(row, oas_bps) - target
    lo, hi = -1000.0, 2000.0
    f_lo, f_hi = f(lo), f(hi)
    if f_lo * f_hi > 0:
        res = minimize_scalar(lambda z: f(z)**2, bounds=(lo, hi), method="bounded")
        return res.x
    else:
        return brentq(f, lo, hi)

In [21]:
# price using forward method + Z-spread
df_bonds["Price_Fwd_Z"] = df_bonds.apply(lambda r: price_callable_forward(r, r["Z_Spread_bps"]), axis=1)
df_bonds[["Ticker", "Price_Fwd_Z"]]

,Ticker,Price_Fwd_Z
0,AMCX,100.102897
1,ANF,107.382899
2,CRL,102.730472
3,HWM,115.210944
4,JWN,103.782090
5,MTCHII,102.647967
6,TFX,104.674575
7,URI,104.238866


In [22]:
# OAS for each bond
df_bonds["OAS_bps"] = df_bonds.apply(solve_oas, axis=1)

In [23]:
df_bonds["Z_minus_OAS_bps"] = df_bonds["Z_Spread_bps"] - df_bonds["OAS_bps"]
## 50 bps condition
df_bonds["Projected_Called"] = df_bonds["Z_minus_OAS_bps"] >= 50.0

## if has_call (having First Call Date present) then we use OAS
df_bonds["Has_Call"] = df_bonds["First Call Date"].notna()

## when its callable use OAS else z spread (here each one has_call)
df_bonds["Comp_Spread_bps"] = np.where(df_bonds["Has_Call"], df_bonds["OAS_bps"], df_bonds["Z_Spread_bps"],)

In [24]:
print("\n=== Spreads & callable pricing (Part 1) ===")
df_bonds[["Ticker", "Price", "Z_Spread_bps", "Price_Fwd_Z", "OAS_bps", "Z_minus_OAS_bps", "Projected_Called","Comp_Spread_bps"]]


=== Spreads & callable pricing (Part 1) ===


,Ticker,Price,Z_Spread_bps,Price_Fwd_Z,OAS_bps,Z_minus_OAS_bps,Projected_Called,Comp_Spread_bps
0,AMCX,100.625000,301.390501,100.102897,293.391358,7.999142,False,293.391358
1,ANF,110.875000,525.442661,107.382899,185.094677,340.347984,True,185.094677
2,CRL,103.125000,266.831906,102.730472,260.384375,6.447531,False,260.384375
3,HWM,115.849998,197.360040,115.210944,180.581287,16.778753,False,180.581287
4,JWN,104.038078,230.406951,103.782090,225.598192,4.808759,False,225.598192
5,MTCHII,103.125000,305.082114,102.647967,297.267121,7.814994,False,297.267121
6,TFX,105.875000,259.155731,104.674575,169.629331,89.526400,True,169.629331
7,URI,105.977997,345.516241,104.238866,137.155322,208.360919,True,137.155322


In [25]:
best_idx = df_bonds["Comp_Spread_bps"].idxmax()
best_bond = df_bonds.loc[best_idx]

print("Cheapest bond (highest comparable spread):")
print(best_bond)

Cheapest bond (highest comparable spread):
Ticker                            MTCHII
Coupon                             4.625
Maturity             2028-06-01 00:00:00
Price                            103.125
Yield                              3.763
Duration                           4.404
Date                 2021-06-23 00:00:00
First Call Date      2023-06-01 00:00:00
First Call Price                 102.313
Second Call Date     2024-06-01 00:00:00
Second Call Price                101.156
Third Call Date      2025-06-01 00:00:00
Third Call Price                   100.0
Z_Spread_bps                  305.082114
Price_Fwd_Z                   102.647967
OAS_bps                       297.267121
Z_minus_OAS_bps                 7.814994
Projected_Called                   False
Has_Call                            True
Comp_Spread_bps               297.267121
Name: 5, dtype: object


## PART 2

In [27]:
def empirical_duration(ticker):
    sub = df_hist[df_hist["Ticker"] == ticker].sort_values("Date")
    price = sub["Price"].values
    y_dec = sub["Yield"].values / 100.0  # convert to decimal yields
    # Compute dp and dy
    dp = (price[1:] - price[:-1]) / price[:-1]
    dy = (y_dec[1:] - y_dec[:-1])
    # Reshape dy for sklearn (needs 2D input)
    dy_reshaped = dy.reshape(-1, 1)

    # Fit linear regression: dp = α + β * dy
    model = LinearRegression().fit(dy_reshaped, dp)
    beta = model.coef_[0]        # regression slope
    D_emp = -beta                # duration = -β
    return D_emp

In [59]:
df_bonds["Emp_Duration"] = df_bonds["Ticker"].apply(empirical_duration)
df_bonds["Duration_Diff"] = df_bonds["Emp_Duration"] - df_bonds["Duration"]
different_exposure = df_bonds["Duration_Diff"].abs() > 0.5
df_bonds["Different_Exposure"] = different_exposure

In [61]:
print("\n=== Empirical durations (Part 2) ===")
df_bonds[["Ticker", "Duration", "Emp_Duration", "Duration_Diff", "Different_Exposure"]]


=== Empirical durations (Part 2) ===


,Ticker,Duration,Emp_Duration,Duration_Diff,Different_Exposure
0,AMCX,5.548,5.035381,-0.512619,True
1,ANF,0.988,0.947493,-0.040507,False
2,CRL,4.246,3.793248,-0.452752,False
3,HWM,3.364,2.336169,-1.027831,True
4,JWN,5.028,4.007812,-1.020188,True
5,MTCHII,4.404,4.350843,-0.053157,False
6,TFX,2.280,1.832810,-0.447190,False
7,URI,1.012,0.429646,-0.582354,True


In [67]:
print("Bonds with different duration exposure than reported: ") 
print(df_bonds[different_exposure]["Ticker"])

Bonds with different duration exposure than reported: 
0    AMCX
3     HWM
4     JWN
7     URI
Name: Ticker, dtype: object
